<a href="https://colab.research.google.com/github/YasserJEP/TDA-BifurcacionesHopf/blob/main/Teaspoon_modificado/autonomous_systems_flows_rk4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Modificación del código fuente de la librería Teaspoon para incluir los tres sistemas dinámicos estudiados

In [ ]:

import numpy as np

def rk4_step(f, y, t, dt):
    k1 = np.asarray(f(y, t))
    k2 = np.asarray(f(y + dt/2 * k1, t + dt/2))
    k3 = np.asarray(f(y + dt/2 * k2, t + dt/2))
    k4 = np.asarray(f(y + dt * k3, t + dt))
    return y + dt/6 * (k1 + 2*k2 + 2*k3 + k4)

def rk4_integrate(f, y0, t):
    y = np.zeros((len(t), len(y0)))
    y[0] = y0
    dt = t[1] - t[0]
    for i in range(1, len(t)):
        y[i] = rk4_step(f, y[i-1], t[i-1], dt)
    return y

def autonomous_systems_flows_rk4(system, dynamic_state=None, L=None, fs=None,
                                 SampleSize=None, parameters=None,
                                 InitialConditions=None):
    if fs is None:
        fs = 100
    if SampleSize is None:
        SampleSize = 2000
    if L is None:
        L = 100.0
    t = np.linspace(0, L, int(L * fs))

    if system == 'lorenz':
        if parameters is not None and len(parameters) == 3:
            rho, sigma, beta = parameters
        else:
            rho, sigma, beta = 24.0, 10.0, 8.0 / 3.0

        def lorenz(state, t):
            x, y, z = state
            dxdt = sigma * (y - x)
            dydt = x * (rho - z) - y
            dzdt = x * y - beta * z
            return dxdt, dydt, dzdt

        if InitialConditions is None:
            InitialConditions = [1.0, 1.0, 1.0]

        states = rk4_integrate(lorenz, InitialConditions, t)

        #last_seconds = 20
        #N = int(last_seconds * fs)

        #ts = [states[-N:, 0], states[-N:, 1], states[-N:, 2]]
        #t  = t[-N:]
        ts = [(states[:, 0])[-SampleSize:], (states[:, 1])[-SampleSize:], (states[:, 2])[-SampleSize:]]
        t = t[-SampleSize:]


    elif system == 'oscilador_Belousov_Zhabotinski_2D':
        if parameters is not None and len(parameters) == 2:
            a, b = parameters
        else:
            a, b = 0.5, 0.5  # Valores por defecto

        def oscilador_Belousov_Zhabotinski_2D(state, t):
            x, y = state
            dxdt = a - x - (4 * x * y) / (1 + x**2)
            dydt = b * x * (1 - y / (1 + x**2))
            return dxdt, dydt

        if InitialConditions is None:
            InitialConditions = [1.0, 1.0]  # Condiciones iniciales por defecto

        states = rk4_integrate(oscilador_Belousov_Zhabotinski_2D, InitialConditions, t)
        ts = [(states[:, 0])[-SampleSize:], (states[:, 1])[-SampleSize:]]
        t = t[-SampleSize:]

    elif system == 'hopf_normal':
        # Forma normal de Hopf supercrítica
        # μ < 0: punto fijo estable
        # μ = 0: bifurcación de Hopf
        # μ > 0: ciclo límite estable

        if parameters is not None and len(parameters) == 2:
            mu, omega = parameters
        else:
            mu, omega = -0.1, 1.0   # valor por defecto (antes de Hopf)

        def hopf_normal(state, t):
            x, y = state
            r2 = x**2 + y**2
            dxdt = mu * x - omega * y - r2 * x
            dydt = omega * x + mu * y - r2 * y
            return dxdt, dydt

        if InitialConditions is None:
            InitialConditions = [0.1, 0.1]

        states = rk4_integrate(hopf_normal, InitialConditions, t)

        ts = [(states[:, 0])[-SampleSize:], (states[:, 1])[-SampleSize:]]
        t = t[-SampleSize:]


    else:
        raise ValueError("El sistema solicitado no está disponible. Usa 'lorenz', 'oscilador_Belousov_Zhabotinski_2D' o 'hopf_normal'.")

    return t, ts